# Hugging Face Basics

Two quick pipelines to confirm your environment can pull pretrained models from the Hugging
Face Hub and run them: sentiment classification and text generation.

Unlike the `pytorch-scientific-ml` cookbooks, this one needs internet access on first use -
the models below get downloaded and cached under `HF_HOME` (LabPod points this at
`/work/.hf-cache`, so it survives stopping and starting the workspace - only the first run
downloads anything).

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
device = 0 if torch.cuda.is_available() else -1  # transformers pipeline device convention

## Sentiment classification

`distilbert-base-uncased-finetuned-sst-2-english` - a small model fine-tuned on the SST-2
movie review dataset. Worth knowing up front: because it's trained specifically on movie
reviews, it can misclassify sentences far outside that domain (try your own out-of-domain
example after this cell and see).

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

examples = [
    "I love this movie!",
    "This movie is terrible.",
    "This is the best day of my life.",
]
for text in examples:
    result = classifier(text)[0]
    print(f"{result['label']:>8}  ({result['score']:.3f})  {text}")

## Text generation

`distilgpt2` - a small GPT-2 variant. `set_seed` makes the sampled output reproducible.

In [ ]:
from transformers import set_seed

set_seed(0)
generator = pipeline("text-generation", model="distilgpt2", device=device)

output = generator(
    "The best way to learn machine learning is",
    max_new_tokens=30,
    num_return_sequences=1,
    do_sample=True,
)
print(output[0]["generated_text"])

## Next steps

- Swap in a larger model (check its size on the Hub page first - it needs to fit in your
  workspace's GPU memory / fractional GPU allocation).
- Try `pipeline("summarization")` or `pipeline("question-answering")` - same pattern, different
  task.
- For fine-tuning rather than just inference, look at `transformers.Trainer` or the `peft`
  library (LoRA) - add them to `template/context/requirements.txt` and rebuild.